# Training (Fine-Tuning) TinyLLama with Generated MCQs

## Validate & Format Training Data

In [5]:
import json

with open("../data/medquad_gen_mcqs/generated_mcqs.jsonl") as f:
    raw_data = [json.loads(line) for line in f]

validated_mcqs = []
for row in raw_data:
    # Validate
    invalid_choices = "A. A skin condition\nB. A viral infection\nC. A lung disease that causes airway inflammation\nD. A type of heart failure"
    if row["mcq_choices"] == invalid_choices and row["mcq_answer"] == "C":
        continue
    # Format
    prompt = row["question"].strip() + "\n" + row["mcq_choices"].strip()
    completion = row["mcq_reason"].strip() + "\n<answer>" + row["mcq_answer"].strip() + "</answer>"
    validated_mcqs.append({"prompt": prompt, "completion": completion})

# Save the formatted dataset
with open("validated_mcqs.jsonl", "w") as f:
    for item in validated_mcqs:
        f.write(json.dumps(item) + "\n")

## Split Data into Train/Test

In [7]:
import json
import random

# Load data
with open("../data/validated_mcqs.jsonl", "r") as f:
    records = [json.loads(line) for line in f]

# Shuffle
random.seed(42)
random.shuffle(records)

# Split: 80% train, 20% test
split_idx = int(0.8 * len(records))
train_set = records[:split_idx]
test_set = records[split_idx:]

# Save to files
with open("mcq_train.jsonl", "w") as f:
    for entry in train_set:
        f.write(json.dumps(entry) + "\n")

with open("mcq_test.jsonl", "w") as f:
    for entry in test_set:
        f.write(json.dumps(entry) + "\n")

print(f"Split complete: {len(train_set)} train, {len(test_set)} test")


Split complete: 492 train, 123 test
